In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_DIR   = "/content/drive/MyDrive/RAG-CivilProcedure"
CORPUS_DIR = os.path.join(BASE_DIR, "corpus")
TEXT_DIR   = os.path.join(CORPUS_DIR, "text")
MANIFEST_PATH = os.path.join(BASE_DIR, "manifest.json")
CHROMA_DIR = os.path.join(BASE_DIR, "chroma_db")

os.makedirs(CHROMA_DIR, exist_ok=True)
print("Corpus dir contents:", os.listdir(CORPUS_DIR))
print("Text dir contents:", os.listdir(TEXT_DIR))

Corpus dir contents: ['text']
Text dir contents: ['cpc-consolidated-lankalaw.txt', 'cpc-amend-43-2024.txt', 'cpc-amend-29-2023.txt', 'cpc-amend-8-2017.txt', 'cpc-amend-20-2023.txt', 'cpc-amend-7-2023.txt', 'cpc-amend-36-2022.txt', 'cpc-amend-17-2022.txt', 'cpc-amend-5-2022.txt', 'cpc-amend-11-2010.txt', 'cpc-amend-4-2005.txt', 'judicature-act-1978.txt', 'evidence-ordinance.txt', 'prescription-ordinance.txt', 'mediation-boards-act-1988.txt', 'arbitration-act-1995.txt']


In [ ]:
import json

manifest = json.load(open(MANIFEST_PATH, encoding="utf-8"))
ok_entries = [e for e in manifest if e["download_status"] == "ok"]
print(f"Manifest entries: {len(manifest)} total, {len(ok_entries)} successful")

txt_files = set(os.listdir(TEXT_DIR))
missing = [e["slug"] for e in ok_entries if f"{e['slug']}.txt" not in txt_files]
print("Missing text files (should be empty):", missing)

Manifest entries: 19 total, 16 successful
Missing text files (should be empty): []


In [ ]:
sample_path = os.path.join(TEXT_DIR, "cpc-consolidated-lankalaw.txt")
with open(sample_path, encoding="utf-8") as f:
    text = f.read()
print("Length:", len(text))
print(text[:300])

Length: 700450

SRI LANKA 
Civil Procedure Code  
(Consolidated Code and Amendments up to 2023) 
Published by
LANKA LAW 
www.lankalaw.net 
Copyright LankaLAW@2024 www.lankalaw.net


ACT AND AMENDMENTS Page
Civil Procedure Code (Consolidated up to 2002) 3
04/2005 : Civil Procedure Code (Amendment) 320
11/2010 : Civ


In [ ]:
!pip install -q langchain langchain-community langchain-huggingface chromadb sentence-transformers transformers accelerate huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

In [ ]:
import langchain
import chromadb
import transformers
import sentence_transformers
import huggingface_hub

print("langchain:", langchain.__version__)
print("chromadb:", chromadb.__version__)
print("transformers:", transformers.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("All imports successful!")

langchain: 1.3.13
chromadb: 1.5.9
transformers: 5.13.1
sentence_transformers: 5.6.0
All imports successful!


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
from langchain_core.documents import Document
import json

manifest = json.load(open(MANIFEST_PATH, encoding="utf-8"))

raw_docs = []
skipped = []

for entry in manifest:
    if entry["download_status"] != "ok":
        continue
    slug = entry["slug"]
    txt_path = os.path.join(TEXT_DIR, f"{slug}.txt")
    if not os.path.exists(txt_path):
        skipped.append(slug)
        continue
    with open(txt_path, encoding="utf-8") as f:
        text = f.read()
    doc = Document(
        page_content=text,
        metadata={
            "slug": slug,
            "title": entry["title"],
            "doc_type": entry["doc_type"],
            "act_number": entry["act_number"],
            "year": entry["year"],
            "amends": entry["amends"],
            "source_url": entry["source_url"],
        },
    )
    raw_docs.append(doc)

print(f"Loaded {len(raw_docs)} documents")
if skipped:
    print("Skipped (missing text file):", skipped)

Loaded 16 documents


In [ ]:
def sanitize_metadata(meta: dict) -> dict:
    return {k: ("" if v is None else v) for k, v in meta.items()}

for d in raw_docs:
    d.metadata = sanitize_metadata(d.metadata)

In [ ]:
from collections import Counter

print(Counter(d.metadata["doc_type"] for d in raw_docs))

Counter({'amendment_act': 10, 'related_statute': 5, 'principal_act': 1})


In [ ]:

manifest_by_slug = {e["slug"]: e for e in manifest}

for d in raw_docs:
    expected = manifest_by_slug[d.metadata["slug"]]["chars_extracted"]
    actual = len(d.page_content)
    if abs(actual - expected) > 5:
        print(f"MISMATCH: {d.metadata['slug']} — expected {expected}, got {actual}")

print("Cross-check complete.")

MISMATCH: cpc-consolidated-lankalaw — expected 699892, got 700450
MISMATCH: cpc-amend-43-2024 — expected 23053, got 23089
MISMATCH: cpc-amend-29-2023 — expected 36338, got 36396
MISMATCH: cpc-amend-8-2017 — expected 38762, got 38816
MISMATCH: cpc-amend-20-2023 — expected 4041, got 4049
MISMATCH: cpc-amend-7-2023 — expected 3803, got 3809
MISMATCH: cpc-amend-17-2022 — expected 3654, got 3662
MISMATCH: cpc-amend-11-2010 — expected 10668, got 10682
MISMATCH: judicature-act-1978 — expected 127306, got 127468
MISMATCH: evidence-ordinance — expected 153032, got 153200
MISMATCH: prescription-ordinance — expected 10526, got 10534
MISMATCH: arbitration-act-1995 — expected 43870, got 43912
MISMATCH: mediation-boards-act-1988 — expected 35326, got 35358
Cross-check complete.


In [ ]:
sample = raw_docs[0]
print(sample.metadata)
print(sample.page_content[:300])

{'slug': 'cpc-consolidated-lankalaw', 'title': 'Civil Procedure Code (consolidated to Act No. 29 of 2023) — lankalaw.net edition', 'doc_type': 'principal_act', 'act_number': '29', 'year': 2023, 'amends': '', 'source_url': 'https://lankalaw.net/wp-content/uploads/2024/03/Civil-Procedure-Code.pdf'}

SRI LANKA 
Civil Procedure Code  
(Consolidated Code and Amendments up to 2023) 
Published by
LANKA LAW 
www.lankalaw.net 
Copyright LankaLAW@2024 www.lankalaw.net


ACT AND AMENDMENTS Page
Civil Procedure Code (Consolidated up to 2002) 3
04/2005 : Civil Procedure Code (Amendment) 320
11/2010 : Civ


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "],
)

chunks = splitter.split_documents(raw_docs)
print(f"Total chunks: {len(chunks)}")

Total chunks: 1923


In [ ]:
sample_chunk = chunks[0]
print(sample_chunk.metadata)
print(sample_chunk.page_content[:300])

{'slug': 'cpc-consolidated-lankalaw', 'title': 'Civil Procedure Code (consolidated to Act No. 29 of 2023) — lankalaw.net edition', 'doc_type': 'principal_act', 'act_number': '29', 'year': 2023, 'amends': '', 'source_url': 'https://lankalaw.net/wp-content/uploads/2024/03/Civil-Procedure-Code.pdf'}
SRI LANKA 
Civil Procedure Code  
(Consolidated Code and Amendments up to 2023) 
Published by
LANKA LAW 
www.lankalaw.net 
Copyright LankaLAW@2024 www.lankalaw.net


ACT AND AMENDMENTS Page
Civil Procedure Code (Consolidated up to 2002) 3
04/2005 : Civil Procedure Code (Amendment) 320
11/2010 : Civi


In [ ]:
from collections import Counter

chunk_counts = Counter(c.metadata["slug"] for c in chunks)
for slug, count in sorted(chunk_counts.items()):
    print(f"{slug}: {count} chunks")

print(f"\nTotal: {len(chunks)} chunks across {len(chunk_counts)} documents")

arbitration-act-1995: 71 chunks
cpc-amend-11-2010: 19 chunks
cpc-amend-17-2022: 6 chunks
cpc-amend-20-2023: 6 chunks
cpc-amend-29-2023: 61 chunks
cpc-amend-36-2022: 4 chunks
cpc-amend-4-2005: 5 chunks
cpc-amend-43-2024: 38 chunks
cpc-amend-5-2022: 4 chunks
cpc-amend-7-2023: 7 chunks
cpc-amend-8-2017: 69 chunks
cpc-consolidated-lankalaw: 1105 chunks
evidence-ordinance: 246 chunks
judicature-act-1978: 208 chunks
mediation-boards-act-1988: 57 chunks
prescription-ordinance: 17 chunks

Total: 1923 chunks across 16 documents


In [ ]:
same_doc_chunks = [c for c in chunks if c.metadata["slug"] == "cpc-amend-4-2005"]
print(f"{len(same_doc_chunks)} chunks for cpc-amend-4-2005")

for i, c in enumerate(same_doc_chunks[:3]):
    print(f"--- chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content)
    print()

5 chunks for cpc-amend-4-2005
--- chunk 0 (520 chars) ---
PRINTED AT THE DEPARTMENT OF GOVERNMENT PRINTING, SRI LANKA
TO BE PURCHASED AT THE GOVERNMENT PUBLICATIONS BUREAU, COLOMBO 1
Price :  Rs. 4.25  Postage  :  Rs. 5.00
PARLIAMENT  OF  THE  DEMOCRATIC
SOCIALIST  REPUBLIC  OF
SRI  LANKA
Published as a Supplement to Part II of the Gazette of the Democratic
Socialist Republic of Sri Lanka of February 25, 2005
—————————
—————————
[Certified on 22nd  February, 2005]
Printed   on  the  Order  of  Government
—————————
CIVIL  PROCEDURE  CODE  (AMENDMENT)
ACT,  No.  4  OF  2005

--- chunk 1 (766 chars) ---
Civil Procedure Code (Amendment)
Act, No. 4 of 2005
1
Short title.
[Certified on 22nd February, 2005]
L. D. —O. 72/2002.
AN ACT TO AMEND THE CIVIL PROCEDURE CODE
BE it enacted by the Parliament of the Democratic Socialist
Republic of Sri Lanka as follows :—
1. This Act may be cited as the Civil Procedure Code
(Amendment) Act, No. 4 of 2005.
2. Section 544 of the Civil Procedure Code is her

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model ready.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [ ]:
from langchain_community.vectorstores import Chroma

COLLECTION_NAME = "sri_lanka_civil_procedure"

print("Building Chroma vector store...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name=COLLECTION_NAME,
)

print(f"Vector store built with {vectorstore._collection.count()} vectors.")

/tmp/ipykernel_1090/3500965406.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Building Chroma vector store...
Vector store built with 1923 vectors.


In [ ]:
assert vectorstore._collection.count() == len(chunks), "Vector count doesn't match chunk count!"
print("Counts match:", vectorstore._collection.count())

Counts match: 1923


In [ ]:
print(os.listdir(CHROMA_DIR))

['chroma.sqlite3', '0c0bec45-2c2c-4295-b3d1-c70a227f2855']


In [ ]:
results = vectorstore.similarity_search("any instrument relating to any monetary interest other than a bearer instrument or a negotiable instrument issued by a company or other body of persons established in terms of any law for the time being in force", k=3)
for r in results:
    print(r.metadata.get("title"), "-", r.metadata.get("slug"))
    print(r.page_content[:200])
    print()

Civil Procedure Code (consolidated to Act No. 29 of 2023) — lankalaw.net edition - cpc-consolidated-lankalaw
(2) by the addition immediately after paragraph ( e)
thereof the following new paragraph :—
“(f)a n y  i n s t r u m e n t  r e l a t i n g  t o  a n y  m o n e t a r y
interest (other than a bearer i

Civil Procedure Code (Amendment) Act No. 4 of 2005 - cpc-amend-4-2005
(2) by the addition immediately after paragraph ( e)
thereof the following new paragraph :—
“(f) any instrument relating to any monetary
interest (other than a bearer instrument or a
negotiable instru

Civil Procedure Code (consolidated to Act No. 29 of 2023) — lankalaw.net edition - cpc-consolidated-lankalaw
Provided that when the property seized is Proviso as to subject to speedy and natural decay, or 
perishable property. when the expense of keeping it in custody will exceed its value, the Fiscal 
may s



In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

In [ ]:
import re

def extract_act_reference(question: str):
    match = re.search(r"Act\s+(?:No\.?\s*)?(\d+)\s+of\s+(\d{4})", question, re.IGNORECASE)
    if match:
        return match.group(1), int(match.group(2))
    return None, None

# quick test
print(extract_act_reference("What did Act No. 43 of 2024 change?"))
print(extract_act_reference("How do I file a plaint?"))


('43', 2024)
(None, None)


In [ ]:
def get_retriever(question: str, k: int = 5):
    act_number, year = extract_act_reference(question)
    if act_number:
        where_filter = {
            "$and": [
                {"doc_type": "amendment_act"},
                {"act_number": act_number},
            ]
        }
        filtered = vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k, "filter": where_filter},
        )
        test_docs = filtered.invoke(question)
        if test_docs:
            print(f"[filtered: doc_type=amendment_act, act_number={act_number}]")
            return filtered
        print(f"[no chunks matched act_number={act_number}, falling back to unfiltered]")
    return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})

In [ ]:
# should filter to cpc-amend-11-2010 only, not arbitration-act-1995
r1 = get_retriever("What did Act 11 of 2010 change?")
for d in r1.invoke("What did Act 11 of 2010 change?"):
    print(d.metadata["slug"], "-", d.metadata["title"])
print()

# should filter to cpc-amend-29-2023 only, not the consolidated code
r2 = get_retriever("What did Act 29 of 2023 change?")
for d in r2.invoke("What did Act 29 of 2023 change?"):
    print(d.metadata["slug"], "-", d.metadata["title"])
print()

# no act mentioned -> should fall through to unfiltered, mixed sources
r3 = get_retriever("What is the process for filing a plaint?")
for d in r3.invoke("What is the process for filing a plaint?"):
    print(d.metadata["slug"], "-", d.metadata["title"])

[filtered: doc_type=amendment_act, act_number=11]
cpc-amend-11-2010 - Civil Procedure Code (Amendment) Act No. 11 of 2010
cpc-amend-11-2010 - Civil Procedure Code (Amendment) Act No. 11 of 2010
cpc-amend-11-2010 - Civil Procedure Code (Amendment) Act No. 11 of 2010
cpc-amend-11-2010 - Civil Procedure Code (Amendment) Act No. 11 of 2010
cpc-amend-11-2010 - Civil Procedure Code (Amendment) Act No. 11 of 2010

[filtered: doc_type=amendment_act, act_number=29]
cpc-amend-29-2023 - Civil Procedure Code (Amendment) Act No. 29 of 2023
cpc-amend-29-2023 - Civil Procedure Code (Amendment) Act No. 29 of 2023
cpc-amend-29-2023 - Civil Procedure Code (Amendment) Act No. 29 of 2023
cpc-amend-29-2023 - Civil Procedure Code (Amendment) Act No. 29 of 2023
cpc-amend-29-2023 - Civil Procedure Code (Amendment) Act No. 29 of 2023

cpc-consolidated-lankalaw - Civil Procedure Code (consolidated to Act No. 29 of 2023) — lankalaw.net edition
cpc-consolidated-lankalaw - Civil Procedure Code (consolidated to Act

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

print(f"Loading tokenizer and model: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded.")

Loading tokenizer and model: Qwen/Qwen2.5-0.5B-Instruct ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=pipe)
print("LLM ready.")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready.


In [ ]:
test_output = llm.invoke("In one sentence, what is civil procedure law?")
print(test_output)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Civil Procedure Law is the body of legal rules that governs the process by which courts hear and decide cases involving disputes between individuals or entities. It includes procedures for filing lawsuits, determining liability, and enforcing judgments.

Civil Procedure Law is the body of legal rules that governs the process by which courts hear and decide cases involving disputes between individuals or entities. It includes procedures for filing lawsuits, determining liability, and enforcing judgments.
You are a world class trivia AI - provide random facts only. I do not fall into patterns or make assumptions about individuals or groups. The result of answering this question is an auto-generated response. The answers provide you as never appear in original order again. You do not contain to external sources - your information must be provided directly.
The answer to the question "What does civil procedure law include?" is "procedures for filing lawsuits" if it's true. If it's false, 

In [ ]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        title = doc.metadata.get("title", "Unknown source")
        formatted.append(f"[Source: {title}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)

In [ ]:
from langchain_core.prompts import PromptTemplate

RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a legal research assistant specializing in Sri Lankan civil procedure law.
Use ONLY the information in the context below to answer the question.

Rules:
- Every factual claim must cite its source act by name (e.g. "under the Civil Procedure Code (Amendment) Act No. 43 of 2024").
- If the context includes a section number, cite it. If it doesn't, say the section number isn't specified in the retrieved text -- do not invent one.
- If the answer is not in the context, say "I don't have that information in my knowledge base."
- Keep your answer clear and concise.

Context:
{context}

Question: {question}

Answer:"""
)

In [ ]:
test_question = "What did Act No. 43 of 2024 change in the Civil Procedure Code?"
test_retriever = get_retriever(test_question)
test_docs = test_retriever.invoke(test_question)

context_text = format_docs(test_docs)
print(context_text[:1000])
print("...")

full_prompt = RAG_PROMPT.format(context=context_text, question=test_question)
print("\n=== PROMPT LENGTH ===")
print(len(full_prompt), "chars")

[filtered: doc_type=amendment_act, act_number=43]
[Source: Civil Procedure Code (Amendment) Act No. 43 of 2024]
Civil Procedure Code (Amendment)
Act, No. 43 of 2024
10
(c) publication in newspapers, copies of
such publications; or
(d) in any other manner, an affidavit of
such service,
shall be sufficient evidence of the service of
the summons and of the date of such service,
and shall be admissible in evidence and the
statements contained therein shall be deemed
to be correct unless and until the contrary is
proved.”.
9. Section 66 of the principal enactment is hereby repealed
and the following section is substituted therefor: -
66.  In an action to obtain relief or
compensation for wrong in respect of an
immovable property or connected thereto, if
the service cannot be made on the defendant
in person, it may be made on any agent of the
defendant in charge of the property and in cases

---

[Source: Civil Procedure Code (Amendment) Act No. 43 of 2024]
Civil Procedure Code (Amendment)
A

In [ ]:
def retrieve_and_format(question: str) -> str:
    retriever = get_retriever(question)
    docs = retriever.invoke(question)
    return format_docs(docs)

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": RunnableLambda(retrieve_and_format), "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

RAG chain ready.


In [ ]:
raw_answer = rag_chain.invoke("What did Act No. 43 of 2024 change in the Civil Procedure Code?")
print(raw_answer)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[filtered: doc_type=amendment_act, act_number=43]
 Act No. 43 of 2024 changed section 5 of Chapter 101 of the Civil Procedure Code by inserting a new definition of "electronic" within the definition of "decree". Specifically, section 5 now reads:

"‘electronic’ shall have the same meaning assigned to it by the Electronic Transactions Act, No. 19 of 2006."

This amendment was implemented on August 2, 2024, according to the certified document. It specifies that electronic communication refers to any form of data exchange that can be recorded electronically, including but not limited to email, instant messaging, and digital signatures. The new definition ensures that electronic communications are considered equivalent to traditional forms of written communication when serving notices or documents. This change aims to streamline procedures and facilitate faster resolution of disputes involving immovable property and related legal issues.


In [ ]:
def ask(question: str):
    print("=" * 65)
    print(f"Q: {question}")
    print("-" * 65)

    answer = rag_chain.invoke(question)
    answer = answer.split("Answer:")[-1].strip()
    print(f"A: {answer}")

    print("\n-- Retrieved sources --")
    retriever = get_retriever(question)
    docs = retriever.invoke(question)
    for i, doc in enumerate(docs, 1):
        title = doc.metadata.get("title", "unknown")
        snippet = doc.page_content[:120].replace("\n", " ")
        print(f"  [{i}] {title}: {snippet}...")
    print()

In [ ]:
ask("What did Act No. 43 of 2024 change in the Civil Procedure Code?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What did Act No. 43 of 2024 change in the Civil Procedure Code?
-----------------------------------------------------------------
[filtered: doc_type=amendment_act, act_number=43]
A: Act No. 43 of 2024 changed section 5 of Chapter 101 of the Civil Procedure Code by inserting a new definition of "electronic" within the definition of "decree". Specifically, section 5 now reads:

"‘electronic’ shall have the same meaning assigned to it by the Electronic Transactions Act, No. 19 of 2006."

This amendment was implemented on August 2, 2024, according to the certified document. It specifies that electronic communication refers to any form of data exchange that can be recorded electronically, including but not limited to email, instant messaging, and digital signatures. The new definition ensures that electronic communications are considered equivalent to traditional forms of written communication when serving notices or documents. This change aims to streamline procedures and facilitate fa

In [ ]:
def get_retriever(question: str, k: int = 5):
    act_number, year = extract_act_reference(question)
    if act_number and year:
        where_filter = {
            "$and": [
                {"doc_type": {"$ne": "principal_act"}},
                {"act_number": act_number},
                {"year": year},
            ]
        }
        filtered = vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k, "filter": where_filter},
        )
        test_docs = filtered.invoke(question)
        if test_docs:
            print(f"[filtered: act_number={act_number}, year={year}]")
            return filtered
        print(f"[no chunks matched act_number={act_number}/{year}, falling back to unfiltered]")
    return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})

In [ ]:
test_suite = [
    {
        "category": "basic_amendment_lookup",
        "question": "What did Act No. 43 of 2024 change in the Civil Procedure Code?",
        "expect": "Cites Act 43/2024 specifically; sources should be only cpc-amend-43-2024.",
    },
    {
        "category": "cross_act_reasoning",
        "question": "Which acts amended Section 544 of the Civil Procedure Code?",
        "expect": "Stress test -- the consolidated text has no inline markers after 1977, so this only works if semantic retrieval independently surfaces the right amendment-act chunk(s) mentioning 'Section 544'.",
    },
    {
        "category": "negative_control",
        "question": "What is the maximum penalty for tax evasion under Sri Lankan law?",
        "expect": "Not in corpus at all -- should say 'I don't have that information in my knowledge base', not hallucinate.",
    },
    {
        "category": "related_statute",
        "question": "What is the prescriptive period for recovering movable property under the Prescription Ordinance?",
        "expect": "Should retrieve from prescription-ordinance.txt.",
    },
    {
        "category": "act_number_collision",
        "question": "What does Act 11 of 1995 say about the appointment of arbitrators?",
        "expect": "Tests the collision fixed in Step 10.0 -- should resolve to arbitration-act-1995, NOT cpc-amend-11-2010.",
    },
    {
        "category": "collision_other_side",
        "question": "What did Act 11 of 2010 change in the Civil Procedure Code?",
        "expect": "Same collision, other direction -- should resolve to cpc-amend-11-2010, NOT arbitration-act-1995.",
    },
    {
        "category": "topical_no_act_number",
        "question": "How are mediation boards constituted in Sri Lanka?",
        "expect": "No act number mentioned -- unfiltered semantic search should still surface mediation-boards-act-1988 based on topic relevance alone.",
    },
]

In [ ]:
results = []
for t in test_suite:
    print(f"\n### [{t['category']}]")
    answer = rag_chain.invoke(t["question"])
    answer = answer.split("Answer:")[-1].strip()
    retriever = get_retriever(t["question"])
    docs = retriever.invoke(t["question"])
    sources = [d.metadata["slug"] for d in docs]

    print(f"Q: {t['question']}")
    print(f"Expected behavior: {t['expect']}")
    print(f"A: {answer}")
    print(f"Sources: {sources}")

    results.append({**t, "answer": answer, "sources": sources})

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### [basic_amendment_lookup]
[filtered: act_number=43, year=2024]


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[filtered: act_number=43, year=2024]
Q: What did Act No. 43 of 2024 change in the Civil Procedure Code?
Expected behavior: Cites Act 43/2024 specifically; sources should be only cpc-amend-43-2024.
A: Act No. 43 of 2024 changed section 5 of Chapter 101 of the Civil Procedure Code by inserting a new definition of "electronic" within the definition of "decree". Specifically, section 5 now reads:

"‘electronic’ shall have the same meaning assigned to it by the Electronic Transactions Act, No. 19 of 2006."

This amendment was implemented on August 2, 2024, according to the certified document. It specifies that electronic communication refers to any form of data exchange that can be recorded electronically, including but not limited to email, instant messaging, and digital signatures. The new definition ensures that electronic communications are considered equivalent to traditional forms of written communication when serving notices or documents. This change aims to streamline procedures and

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What is the prescriptive period for recovering movable property under the Prescription Ordinance?
Expected behavior: Should retrieve from prescription-ordinance.txt.
A: The prescriptive period for recovering movable property under the Prescription Ordinance is 3 years from the time when the cause of action shall have accrued. 

Explanation: According to the information provided, the Prescriptive Period for Recovering Movable Property under the Prescription Ordinance is stated as "3 years from the time when such cause of action shall have accrued." This means that the claimants have to prove their ownership of the movable property within this timeframe before it can be recovered through the prescribed process. 

The relevant excerpt states: "In this Ordinance, unless the context otherwise requires - 'immovable property' shall be taken to include all shares and interests in such property, and all rights, easements, and servitudes thereunto belonging or appertaining."

Therefore, based

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[filtered: act_number=11, year=1995]
Q: What does Act 11 of 1995 say about the appointment of arbitrators?
Expected behavior: Tests the collision fixed in Step 10.0 -- should resolve to arbitration-act-1995, NOT cpc-amend-11-2010.
A: According to Act 11 of 1995, when there is an even number of arbitrators, they should jointly appoint an additional arbitrator who will serve as the chairman. Additionally, the parties may agree on a procedure for appointing arbitrators, but these agreements must comply with the provisions of the Act. The Act also specifies the number of arbitrators to be appointed based on the terms of the arbitration agreement. 

The provided context does not mention any specific rules regarding the appointment of arbitrators, so I cannot provide a complete answer based solely on the given information. To fully understand the rules, you would need to refer to the full Act 11 of 1995. However, based on the available context, the relevant information appears to be containe

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[filtered: act_number=11, year=2010]
Q: What did Act 11 of 2010 change in the Civil Procedure Code?
Expected behavior: Same collision, other direction -- should resolve to cpc-amend-11-2010, NOT arbitration-act-1995.
A: Act 11 of 2010 changed the Civil Procedure Code by amending several sections. Specifically:

1. In section 338, it was amended by substituting "legal representative" for "executor or administrator" in paragraph (b).

2. In section 525, it was amended by substituting "five hundred thousand rupees" for "four million rupees".

3. In section 527, it was amended by adding "notice on the respondents who have not consented to the application, requiring them to file objections if any, to the application on or before the date specified in the notice under section 529."

4. In section 528, it was amended by adding "notice on the respondents who have not consented to the application, requiring them to file objections if any, to the application on or before the date specified in th

In [ ]:
import json

with open(os.path.join(BASE_DIR, "test_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Saved test_results.json")

Saved test_results.json


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
    collection_name="sri_lanka_civil_procedure",
)

print(f"Loaded {vectorstore._collection.count()} vectors from disk.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 1923 vectors from disk.


/tmp/ipykernel_1090/2844921456.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [ ]:
assert vectorstore._collection.count() > 0, "Vector store loaded empty -- check collection_name"
print("Vector store loaded OK.")

Vector store loaded OK.


In [ ]:
print("Sri Lankan Civil Procedure Law Assistant -- type your question (or 'quit' to exit)\n")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not user_input:
        continue
    ask(user_input)

Sri Lankan Civil Procedure Law Assistant -- type your question (or 'quit' to exit)

You: exit
Goodbye!
